# Unit 11 — Number Systems, Bitwise & Number Theory

A light panel stores all of its on/off switches as one integer.
How can you flip switch 3 without touching the rest?
This unit reads the bits inside an integer, changes selected bits, uses masks to represent subsets, and builds several number-theory tools by hand.

## Lesson 1 — Write Numbers in Binary and Hexadecimal

Decimal uses powers of 10, binary uses powers of 2, and hexadecimal uses powers of 16.
The binary number `10110` means `1 · 16 + 0 · 8 + 1 · 4 + 1 · 2 + 0 · 1 = 22`.
Hexadecimal needs six extra digits: `A`, `B`, `C`, `D`, `E`, and `F` represent values 10 through 15.
For example, `2A` means `2 · 16 + 10 = 42`.

## Convert by Hand in Both Directions

To convert a non-negative integer to a base, repeatedly take `number % base` and then update `number = number // base`.
The remainders arrive from right to left, so place each new digit at the front with string concatenation.
Handle zero separately because its repeated-division loop runs zero times.
To parse a digit string, scan left to right and update `value = value * base + digit_value`; this gives every digit its positional value.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    number = int(tokens[0])
    binary_text = tokens[1]
    hexadecimal_text = tokens[2]
    digits = "0123456789ABCDEF"

    if number == 0:
        binary = "0"
    else:
        binary = ""
        remaining = number
        while remaining > 0:
            digit = remaining % 2
            binary = digits[digit] + binary
            remaining = remaining // 2

    if number == 0:
        hexadecimal = "0"
    else:
        hexadecimal = ""
        remaining = number
        while remaining > 0:
            digit = remaining % 16
            hexadecimal = digits[digit] + hexadecimal
            remaining = remaining // 16

    binary_value = 0
    position = 0
    while position < len(binary_text):
        binary_value = binary_value * 2 + int(binary_text[position])
        position = position + 1

    hexadecimal_value = 0
    position = 0
    while position < len(hexadecimal_text):
        character = hexadecimal_text[position]
        digit_value = 0
        while digits[digit_value] != character:
            digit_value = digit_value + 1
        hexadecimal_value = hexadecimal_value * 16 + digit_value
        position = position + 1
    return binary + "\n" + hexadecimal + "\n" + str(binary_value) + "\n" + str(hexadecimal_value)

assert solve("42 10110 2A") == "101010\n2A\n22\n42"
assert solve("0 0 0") == "0\n0\n0\n0"
assert solve("16 10000 10") == "10000\n10\n16\n16"

## Store Switches as Bits

Number bits from right to left starting at index `0`.
The expression `1 << i` moves one on-bit into position `i`, producing a mask for that switch.
Test bit `i` with `x & (1 << i)`, set it with `x | (1 << i)`, clear it with `x & ~(1 << i)`, and flip it with `x ^ (1 << i)`.
The operators `&`, `|`, and `^` combine two bit patterns; `<<` shifts left and `>>` shifts right.

## NOT Needs a Finite Width

Python integers do not stop after a fixed number of bits, so `~5 == -6` instead of a small positive bit pattern.
When a problem gives width `w`, keep only those bits with `~x & ((1 << w) - 1)`.
For width 4, complementing `0101` gives `1010`, which is 10.
Always state and use the width when a complement is required.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    state = int(tokens[0])
    width = int(tokens[1])
    bit_index = int(tokens[2])
    other = int(tokens[3])
    bit_mask = 1 << bit_index
    tested = 0
    if state & bit_mask:
        tested = 1
    set_state = state | bit_mask
    cleared_state = state & ~bit_mask
    flipped_state = state ^ bit_mask
    width_mask = (1 << width) - 1
    complemented = ~state & width_mask
    shifted_left = state << 1
    shifted_right = state >> 1
    shared = state & other
    combined = state | other
    different = state ^ other
    return str(tested) + " " + str(set_state) + " " + str(cleared_state) + " " + str(flipped_state) + "\n" + str(complemented) + " " + str(shifted_left) + " " + str(shifted_right) + "\n" + str(shared) + " " + str(combined) + " " + str(different)

assert solve("5 4 3 6") == "0 13 5 13\n10 10 2\n4 7 3"
assert solve("15 4 0 0") == "1 15 14 14\n0 30 7\n0 15 15"

## Lesson 2 — One Mask per Subset

A mask with `n` bits can describe a subset of `n` items: bit `i` is 1 exactly when item `i` is included.
The masks from `0` through `(1 << n) - 1` cover every subset once, including the empty subset and the full subset.
Iterate with `for mask in range(1 << n)`, then test each item with `mask & (1 << i)`.
This checks `2` to the power `n` subsets, so use it only when `n` is small.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    values = []
    index = 0
    while index < n:
        values.append(int(tokens[index + 2]))
        index = index + 1
    matching_count = 0
    for mask in range(1 << n):
        subset_total = 0
        index = 0
        while index < n:
            if mask & (1 << index):
                subset_total = subset_total + values[index]
            index = index + 1
        if subset_total == target:
            matching_count = matching_count + 1
    return str(matching_count)

assert solve("4 5 8 1 4 10") == "1"
assert solve("3 0 2 4 8") == "1"
assert solve("3 14 2 4 8") == "1"

## Greatest Common Divisor and Least Common Multiple

The greatest common divisor, or GCD, is the largest positive integer dividing both numbers.
Euclid's algorithm repeatedly replaces `(a, b)` with `(b, a % b)` until `b` is zero; the remaining `a` is the GCD.
Use the iterative update `while b: a, b = b, a % b`.
For nonzero inputs, compute the least common multiple with `a // gcd * b`; dividing first keeps the intermediate value smaller.
If either input is zero, define the LCM as zero.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    first = int(tokens[0])
    second = int(tokens[1])
    a = abs(first)
    b = abs(second)
    while b:
        a, b = b, a % b
    greatest = a
    least = 0
    if first != 0 and second != 0:
        least = abs(first) // greatest * abs(second)
    return str(greatest) + " " + str(least)

assert solve("18 24") == "6 72"
assert solve("7 9") == "1 63"
assert solve("12 12") == "12 12"
assert solve("0 15") == "15 0"

## Lesson 3 — Sieve Many Primes at Once

To find every prime through `n`, build a Boolean list and begin by treating every index as possibly prime.
Mark 0 and 1 as not prime.
For each still-prime `p`, cross off `p * p`, `p * p + p`, and later multiples; smaller composite multiples already had a smaller prime factor.
Continue while `p * p <= n`, including the equality so a prime square such as 49 is crossed off.

In [ ]:
def solve(data: str) -> str:
    n = int(data.strip())
    is_prime = []
    index = 0
    while index <= n:
        is_prime.append(True)
        index = index + 1
    if n >= 0:
        is_prime[0] = False
    if n >= 1:
        is_prime[1] = False
    p = 2
    while p * p <= n:
        if is_prime[p]:
            multiple = p * p
            while multiple <= n:
                is_prime[multiple] = False
                multiple = multiple + p
        p = p + 1
    prime_count = 0
    value = 2
    while value <= n:
        if is_prime[value]:
            prime_count = prime_count + 1
        value = value + 1
    return str(prime_count)

assert solve("49") == "15"
assert solve("2") == "1"
assert solve("1") == "0"

## Reduce as You Go

When a problem asks for an answer modulo positive `M`, keep each multiplication small by taking `% M` as you go.
For a huge power, repeated squaring processes the exponent one binary bit at a time: multiply the answer when the current bit is 1, square the base, and shift the exponent right.
Finish every multiplication with `% M`; postponing all multiplication is infeasible when the exponent is enormous.
Moduli in this unit are positive and results are non-negative.
Python also makes a modulo result non-negative for a positive modulus, so `-3 % 5 == 2`.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    base = int(tokens[0])
    modulus = int(tokens[1])
    exponent = int(tokens[2])
    answer = 1 % modulus
    current = base % modulus
    while exponent > 0:
        if exponent & 1:
            answer = answer * current % modulus
        current = current * current % modulus
        exponent = exponent >> 1
    return str(answer)

assert solve("3 7 5") == "5"
assert solve("7 13 1000000000000000000") == "9"
assert solve("9 1 0") == "0"

## A Number-Tools Checklist

Convert bases with repeated remainder and positional value, including the special representation of zero.
For bit operations, number bits from zero and mask every complement to the stated width.
For subsets, visit every mask from the empty mask through the full mask.
Use iterative Euclid for GCD, divide before multiplying for LCM, begin sieve crossing at `p * p`, and reduce modular products after every multiplication.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper.
It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))